# 02 - Two from-scratch DPO runs  (Task 4)

Inputs: **attach the output of notebook 01** (Add Input -> Your Work -> Notebooks -> 01, latest version).

Both runs use the same code (`scripts/train_dpo.py`) and start from the SFT **best** checkpoint; a second frozen copy
of it is the reference. Guide config: beta 0.1, lr 1e-6 fixed, RMSprop, batch 1 x accum 32, clip 1.0, max length 512 /
prompt 256, 1 epoch, eval every 50 steps on 1,000 val pairs, best checkpoint by validation reward accuracy + final.

* Default: runs sequentially with policy on cuda:0 (fp32 + fp16 autocast) and reference on cuda:1 (manual 10.1 tip).
  About 2-2.5 h per run, so ~4.5-5 h total; it fits in one 12 h Kaggle session.
* `PARALLEL_DPO = True`: runs HH on cuda:0 and UF on cuda:1 at the same time (policy + reference share a GPU, ~11-12 GB),
  ~2.5 h total. If you use it, report it as a resource-driven deviation in the report.
* `RUN_TRACKS` lets you split the two runs across two notebook versions if needed.

**Kaggle settings for this notebook:** Accelerator = **GPU T4 x2**, Internet = **On** (needs a phone-verified account),
Persistence = Files only (optional). Edit `REPO_URL` below, then **Save Version -> Save & Run All (Commit)** so the
run continues in the background and all files in `/kaggle/working` are kept as this version's output.

In [ ]:
# ---- EDIT THIS: your GitHub fork of the starter repo with this code pushed to it ----
REPO_URL = "https://github.com/Ibrahim2424777/CS5326-PA2.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/cs5326-pa2"

PARALLEL_DPO = False
RUN_TRACKS = ["hh", "uf"]

In [ ]:
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:  # already cloned in this session -> pick up the latest pushed code
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
from smalign.notebook import sh, start_bg, wait_bg, find_input, copy_run_metadata, zip_dir, disk_report

In [ ]:
# Task 1: install deps WITHOUT reinstalling Kaggle's CUDA-matched PyTorch (manual 5.2)
sh("python -m pip uninstall -y -q torchao || true")
sh('python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml '
   'sentencepiece safetensors pytest matplotlib')
sh("python -m pip install -q -e . --no-deps")

In [ ]:
# Task 1: confirm two T4 GPUs and print package versions
sh("nvidia-smi")
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("WARNING: expected GPU T4 x2 (Settings -> Accelerator). Scripts will fall back to one GPU but run slower.")
disk_report()

In [ ]:
SFT_BEST = os.path.dirname(find_input("outputs/sft/best/config.json"))
DATA = os.path.dirname(find_input("outputs/data/hh_train.jsonl"))
sh(f"cat {SFT_BEST}/checkpoint_state.json | head -c 400; echo; wc -l {DATA}/*.jsonl")

Step-0 evaluation is a sanity check: with policy == reference the val loss must be ln 2 = 0.6931 and reward accuracy 0.

In [ ]:
def dpo_cmd(track, policy_device, reference_device):
    return (f"python scripts/train_dpo.py --config configs/dpo_{track}.yaml --set "
            f"policy_init={SFT_BEST} reference_path={SFT_BEST} "
            f"train_path={DATA}/{track}_train.jsonl val_path={DATA}/{track}_val.jsonl "
            f"policy_device={policy_device} reference_device={reference_device}")

todo = [t for t in RUN_TRACKS if not os.path.exists(f"outputs/dpo_{t}/summary.json")]
print("tracks to run:", todo)
if PARALLEL_DPO and len(todo) == 2:
    jobs = [start_bg(dpo_cmd(t, f"cuda:{i}", f"cuda:{i}"), f"outputs/logs/dpo_{t}.log") for i, t in enumerate(todo)]
    wait_bg(*jobs, poll_seconds=600)
else:
    for t in todo:
        sh(dpo_cmd(t, "cuda:0", "cuda:1"), log_path=f"outputs/logs/dpo_{t}.log")

In [ ]:
sh("cat outputs/dpo_*/summary.json")
sh("python scripts/plot_training.py --dpo " + " ".join(f"outputs/dpo_{t}" for t in RUN_TRACKS) + " --out_dir outputs/report_assets")
from IPython.display import Image, display
for t in RUN_TRACKS:
    display(Image(f"outputs/report_assets/dpo_{t}_curves.png"))
zip_dir("outputs", "/kaggle/working/dpo_run_outputs.zip")
disk_report()
print("DONE. Save this version; attach its output (and notebook 01's) to notebook 03.")